# Gold Layer — Revenue Comparison

Answers the core business question: **for each Amsterdam postcode, is Airbnb
or long-term rental more profitable?**

Revenue normalisation:
- Airbnb: `nightly_price × 365 × 0.65 / 12`  → estimated monthly revenue
  - 65% occupancy is consistent with published Amsterdam Airbnb market averages
- Rental: `total_monthly_eur` (rent + service costs)

Output tables:
1. `revenue_per_listing` — one row per property with both revenue estimates
2. `revenue_per_postcode` — aggregated by PC4, with a `recommendation` column

In [ ]:
import sys
import os

from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("amsterdam-rental-gold").getOrCreate()

try:
    dbutils.fs.ls("/")  # only exists on Databricks
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    nb_path = ctx.notebookPath().get()
    repo_ws_root = "/Workspace" + "/".join(nb_path.split("/")[:-2])
    sys.path.insert(0, f"{repo_ws_root}/src")

    VOLUME_ROOT = "/Volumes/workspace/default/amsterdam_rental"
    OUTPUT_ROOT = f"{VOLUME_ROOT}/output"
except NameError:
    sys.path.insert(0, os.path.abspath("../src"))
    _repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
    OUTPUT_ROOT = f"{_repo}/data/output"

from amsterdam_rental.transforms import airbnb_monthly_revenue

SILVER_PATH  = f"{OUTPUT_ROOT}/silver"
GOLD_PATH    = f"{OUTPUT_ROOT}/gold"
PARQUET_PATH = f"{OUTPUT_ROOT}/parquet"

OCCUPANCY_RATE = 0.65

In [ ]:
airbnb  = spark.read.format("delta").load(f"{SILVER_PATH}/airbnb")
rentals = spark.read.format("delta").load(f"{SILVER_PATH}/rentals")

print(f"Airbnb silver rows : {airbnb.count()}")
print(f"Rentals silver rows: {rentals.count()}")

## Per-listing revenue estimates

In [ ]:
# Airbnb monthly revenue per listing
airbnb_revenue = (
    airbnb
    .filter(F.col("pc4").isNotNull() & F.col("nightly_price_eur").isNotNull())
    .withColumn(
        "estimated_monthly_revenue_eur",
        F.round(F.col("nightly_price_eur") * 365 * OCCUPANCY_RATE / 12, 2)
    )
    .withColumn("source", F.lit("airbnb"))
    .select(
        F.col("pc4"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("room_type").alias("property_type"),
        F.col("nightly_price_eur"),
        F.col("estimated_monthly_revenue_eur"),
        F.col("source"),
    )
)
print(f"Airbnb revenue rows: {airbnb_revenue.count()}")

In [ ]:
# Rental monthly revenue per listing (actual rent + costs)
rental_revenue = (
    rentals
    .filter(F.col("pc4").isNotNull() & F.col("total_monthly_eur").isNotNull())
    .withColumn("estimated_monthly_revenue_eur", F.round(F.col("total_monthly_eur"), 2))
    .withColumn("source", F.lit("rental"))
    .select(
        F.col("pc4"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("property_type"),
        F.lit(None).cast(T.DoubleType()).alias("nightly_price_eur"),
        F.col("estimated_monthly_revenue_eur"),
        F.col("source"),
    )
)
print(f"Rental revenue rows: {rental_revenue.count()}")

In [ ]:
revenue_per_listing = airbnb_revenue.unionByName(rental_revenue)

(
    revenue_per_listing
    .write.format("delta").mode("overwrite")
    .save(f"{GOLD_PATH}/revenue_per_listing")
)
print(f"revenue_per_listing written: {revenue_per_listing.count()} rows")

## Per-postcode aggregation

In [ ]:
airbnb_by_pc4 = (
    airbnb_revenue
    .groupBy("pc4")
    .agg(
        F.round(F.avg("estimated_monthly_revenue_eur"), 2).alias("airbnb_avg_monthly_eur"),
        F.count("*").alias("airbnb_listing_count"),
    )
)

rental_by_pc4 = (
    rental_revenue
    .groupBy("pc4")
    .agg(
        F.round(F.avg("estimated_monthly_revenue_eur"), 2).alias("rental_avg_monthly_eur"),
        F.count("*").alias("rental_listing_count"),
    )
)

revenue_per_postcode = (
    airbnb_by_pc4
    .join(rental_by_pc4, on="pc4", how="outer")
    .withColumn(
        "airbnb_premium_eur",
        F.round(F.col("airbnb_avg_monthly_eur") - F.col("rental_avg_monthly_eur"), 2)
    )
    .withColumn(
        "recommendation",
        F.when(F.col("airbnb_avg_monthly_eur") > F.col("rental_avg_monthly_eur"), "airbnb")
         .when(F.col("rental_avg_monthly_eur") > F.col("airbnb_avg_monthly_eur"), "rental")
         .otherwise("neutral")
    )
    .orderBy(F.col("airbnb_premium_eur").desc())
)

print(f"Postcodes with data: {revenue_per_postcode.count()}")
revenue_per_postcode.show(20)

In [ ]:
(
    revenue_per_postcode
    .write.format("delta").mode("overwrite")
    .save(f"{GOLD_PATH}/revenue_per_postcode")
)
print("revenue_per_postcode written.")

## Export as Parquet (required deliverable)

In [ ]:
for name, df in [
    ("revenue_per_listing",   revenue_per_listing),
    ("revenue_per_postcode",  revenue_per_postcode),
    ("airbnb_silver",         airbnb),
    ("rentals_silver",        rentals),
]:
    df.write.mode("overwrite").parquet(f"{PARQUET_PATH}/{name}")
    print(f"  {name} exported")

print("Parquet export complete.")

## Gold validation checks

In [ ]:
gold = spark.read.format("delta").load(f"{GOLD_PATH}/revenue_per_postcode")

# Every postcode should have a recommendation
missing_rec = gold.filter(F.col("recommendation").isNull()).count()
assert missing_rec == 0, f"{missing_rec} postcodes without recommendation"

# Revenue values should be positive where present
bad_airbnb = gold.filter(F.col("airbnb_avg_monthly_eur") <= 0).count()
bad_rental = gold.filter(F.col("rental_avg_monthly_eur") <= 0).count()
assert bad_airbnb == 0, f"{bad_airbnb} postcodes with non-positive Airbnb revenue"
assert bad_rental == 0, f"{bad_rental} postcodes with non-positive rental revenue"

print("Gold validation passed.")
gold.show(10)